# 5. Representatividade e análise bibliométrica (A9)

In [ ]:
#@title Preparação (rode esta célula primeiro) { display-mode: "form" }
REPOSITORIO = "https://github.com/Juliaalv/curso_proknow.git"
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/curso_proknowc"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPOSITORIO, "/content/curso_proknowc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/curso_proknowc"], check=True)
    sys.path.insert(0, "/content/curso_proknowc/src")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from curso_proknowc import colab

pastas = colab.preparar()
print("Tudo pronto. Pasta do curso:", pastas.raiz)

In [ ]:
#@title Entrada { display-mode: "form" }
#@markdown **exemplo do curso**: começa do resultado já pronto do exemplo (curtailment de eólica e solar).
#@markdown **meus arquivos**: começa do resultado que você salvou no notebook anterior.
#@markdown **EMAIL**: o seu e-mail, usado nas consultas ao OpenAlex e ao Unpaywall (o Unpaywall exige).
FONTE = "exemplo do curso" #@param ["exemplo do curso", "meus arquivos"]
EMAIL = "" #@param {type:"string"}
portfolio = colab.carregar(pastas, FONTE, "04_portfolio", notebook="05")
print(f"{len(portfolio)} artigos carregados.")

## Teste de representatividade

Último passo da seleção: as referências mais citadas pelos artigos do portfólio. As que estiverem
alinhadas ao seu tema entram no portfólio como "complementares".

In [ ]:
#@title Referências mais citadas pelo portfólio { display-mode: "form" }
#@markdown Consulta o OpenAlex (precisa de internet).
BUSCAR_REFERENCIAS = True #@param {type:"boolean"}
QUANTAS = 20 #@param {type:"integer"}
from curso_proknowc import bibliometria
if BUSCAR_REFERENCIAS:
    portfolio, avisos = bibliometria.obter_referencias(portfolio, email=EMAIL or None)
    for aviso in avisos:
        print("•", aviso)
mais_citadas, avisos = bibliometria.referencias_mais_citadas(portfolio, QUANTAS, email=EMAIL or None)
for aviso in avisos:
    print("•", aviso)
if len(mais_citadas):
    display(mais_citadas.drop(columns="alinhada"))
else:
    print("Nenhuma referência encontrada. Marque BUSCAR_REFERENCIAS para consultar as referências no OpenAlex.")

In [ ]:
#@title Incorporar as referências alinhadas e salvar { display-mode: "form" }
BAIXAR_CSV = False #@param {type:"boolean"}
#@markdown Copie da tabela acima os códigos (coluna openalex_id) das referências alinhadas ao seu tema, separados por ponto e vírgula.
ALINHADAS = "" #@param {type:"string"}
marcadas = {c.strip() for c in ALINHADAS.split(";") if c.strip()}
mais_citadas["alinhada"] = mais_citadas["openalex_id"].isin(marcadas)
portfolio_final = bibliometria.incorporar_complementares(portfolio, mais_citadas)
incluidas = len(portfolio_final) - len(portfolio)
colab.registrar_etapa(pastas, "05", "Teste de representatividade", len(portfolio), portfolio.iloc[0:0], incluidos=incluidas)
colab.salvar(pastas, portfolio_final, "05_portfolio_final", baixar=BAIXAR_CSV)
print(f"{incluidas} referência(s) incorporada(s). Portfólio final: {len(portfolio_final)} artigos.")

## Análise bibliométrica

In [ ]:
#@title Periódicos, autores, palavras-chave e anos { display-mode: "form" }
QUANTOS_NO_GRAFICO = 10 #@param {type:"integer"}
for coluna in ["periodico", "autores", "palavras_chave"]:
    bibliometria.grafico_frequencias(portfolio_final, coluna, QUANTOS_NO_GRAFICO)
    plt.show()
bibliometria.grafico_anos(portfolio_final)
plt.show()